# 12 Causal Inference — Reference Solutions

Complete solutions for the causal inference exercises based on the Songbai Nursing Home Legionella cluster investigation.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statsmodels.formula.api as smf

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## Question 1: Attributable Risk of Hydrotherapy Exposure

In [ ]:
# Hydrotherapy exposure
hydro_exp = df[df["hydrotherapy_use"] == 1]
hydro_unexp = df[df["hydrotherapy_use"] == 0]

risk_hydro_exp = hydro_exp["infected"].mean()
risk_hydro_unexp = hydro_unexp["infected"].mean()
risk_total = df["infected"].mean()

AR_hydro = risk_hydro_exp - risk_hydro_unexp
PAR_hydro = risk_total - risk_hydro_unexp
PAR_pct_hydro = PAR_hydro / risk_total * 100

print("=== Hydrotherapy Exposure ===")
print(f"Attack rate among users: {risk_hydro_exp:.1%}")
print(f"Attack rate among non-users: {risk_hydro_unexp:.1%}")
print(f"AR = {AR_hydro:.3f}")
print(f"PAR% = {PAR_pct_hydro:.1f}%")

# Shower exposure (for comparison)
shower_exp = df[df["shower_use"] == 1]
shower_unexp = df[df["shower_use"] == 0]
risk_sh_exp = shower_exp["infected"].mean()
risk_sh_unexp = shower_unexp["infected"].mean()
AR_shower = risk_sh_exp - risk_sh_unexp
PAR_shower = risk_total - risk_sh_unexp
PAR_pct_shower = PAR_shower / risk_total * 100

print(f"\n=== Shower Exposure (comparison) ===")
print(f"AR = {AR_shower:.3f}")
print(f"PAR% = {PAR_pct_shower:.1f}%")

print(f"\n=== Comparison ===")
if abs(AR_shower) > abs(AR_hydro):
    print("→ Shower exposure has the larger AR, contributing more to infection")
else:
    print("→ Hydrotherapy exposure has the larger AR")

print("\n→ AR represents 'the amount of risk that could be removed by eliminating the exposure, if the causal relationship holds'")
print("→ Assumptions: (1) the causal relationship holds (2) no confounding (3) the exposure is removable")

## Question 2: Changing the DiD Intervention Date

In [ ]:
cases = df[df["infected"] == 1].copy()
all_dates = pd.date_range("2026-01-12", "2026-01-28", freq="D")

# Treated group / control group
treated_mask = (cases["floor"].isin([2, 3])) & (cases["wing"] == "B")
treated_daily = cases[treated_mask].groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)
control_daily = cases[~treated_mask].groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# Compare different intervention dates
for cutoff in ["2026-01-22", "2026-01-25"]:
    panel = pd.DataFrame({
        "date": list(all_dates) * 2,
        "treated": [1] * len(all_dates) + [0] * len(all_dates),
        "daily_cases": list(treated_daily.values) + list(control_daily.values),
    })
    panel["post"] = (panel["date"] >= cutoff).astype(int)

    model = smf.ols("daily_cases ~ treated + post + treated:post", data=panel).fit()
    coef = model.params["treated:post"]
    pval = model.pvalues["treated:post"]

    print(f"Intervention date = {cutoff}: treated:post = {coef:.3f}, p = {pval:.4f}")

print("\n→ Changing the intervention date affects the DiD result")
print("→ Reason: the number of observation days before and after the intervention differs, and so does the distribution of cases")
print("→ The intervention date must be based on the actual event (disinfection really happened); it cannot be picked arbitrarily")
print("→ Cherry-picking an intervention date to produce a significant result = p-hacking")

## Question 3 (Challenge): Demonstrating Collider Bias

In [ ]:
from epi_learning import risk_ratio

# RR for the whole sample
ct_all = pd.crosstab(df["shower_use"], df["infected"])
rr_all = risk_ratio(ct_all.iloc[1, 1], ct_all.iloc[1].sum(), ct_all.iloc[0, 1], ct_all.iloc[0].sum())
print(f"=== Whole-sample RR (shower → infected) ===")
print(f"RR = {rr_all:.3f}")

# Restrict to hospitalized patients
hosp = df[df["hospitalized"] == 1].copy()
print(f"\nHospitalized patients: {len(hosp)}")
print(f"shower_use distribution among hospitalized: {hosp['shower_use'].value_counts().to_dict()}")
print(f"infected distribution among hospitalized: {hosp['infected'].value_counts().to_dict()}")

# Are all hospitalized patients infected?
if hosp["infected"].nunique() == 1:
    print("\n→ All hospitalized patients are infected (infected=1), so RR cannot be computed")
    print("→ This is precisely the extreme case of collider bias!")
    print("→ Because only people who are infected and severe get hospitalized")
    print("→ Among hospitalized patients, the relationship between shower_use and infected is distorted")
else:
    ct_hosp = pd.crosstab(hosp["shower_use"], hosp["infected"])
    rr_hosp = risk_ratio(ct_hosp.iloc[1, 1], ct_hosp.iloc[1].sum(), ct_hosp.iloc[0, 1], ct_hosp.iloc[0].sum())
    print(f"RR among hospitalized = {rr_hosp:.3f}")
    print(f"Whole-sample RR = {rr_all:.3f}")
    print(f"\n→ The RR changed after restricting to hospitalized patients!")
    print("→ This is collider bias")

print("\n=== Explaining Collider Bias ===")
print("hospitalized ← severity ← infection")
print("hospitalized ← infection")
print("→ hospitalized is a collider, jointly affected by severity and infection")
print("→ Conditioning on the collider (looking only at hospitalized patients) = opening a spurious path")
print("→ Result: among hospitalized patients, the relationship between shower_use and infection is distorted")

### Interpretation

- **AR/PAR**: the attributable risk differs between hydrotherapy and showering, reflecting the contributions of different exposure routes. Showering is the main route that generates Legionella aerosols
- **DiD intervention date**: the result is sensitive to the intervention date. The correct approach is to use the actual intervention date, not to pick the most significant one after the fact
- **Collider**: analyzing only hospitalized patients = conditioning on a collider, which introduces selection bias. This is a common trap in observational studies
- **Limits of causal inference**: with observational data, we can never be fully certain about causation. DAGs and statistical methods can only help us identify and reduce bias—they cannot eliminate all unobserved confounders